In [ ]:
from google.colab import drive
import os
import sys

# Mount Drive
drive.mount('/content/drive')

# Define the project folder (change the path if different)
PROJECT_ROOT = '/content/drive/MyDrive/Speach_Emotion_Recognition'
os.chdir(PROJECT_ROOT)

# Update the cartridge to the system to allow imports (e.g. from models.model import...)
sys.path.append(PROJECT_ROOT)

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Check if files exist
test_path = 'dataset/augmentedDataset_9K_RandomSplit/X_train.npy'
if os.path.exists(test_path):
    print(" Files found successfully on Drive!")
else:
    print(" Warning: Path not found. Check the folder on Drive.")

In [ ]:
# @title Experiment Configuration
esperimento = "Random Split (Augmented)" # @param ["Actor Split (Augmented)", "Random Split (Augmented)", "Pure Dataset"]

if esperimento == "Actor Split (Augmented)":
    data_path = 'dataset/augmentedDataset_9K_ActorSplit'
    results_folder = 'res_ActorSplit'
elif esperimento == "Random Split (Augmented)":
    data_path = 'dataset/augmentedDataset_9K_RandomSplit'
    results_folder = 'res_RandomSplit'
else:
    data_path = 'dataset/pureDataset_RandomSplit'
    results_folder = 'res_PureDataset'

results_path = os.path.join('checkpoints', results_folder)
os.makedirs(results_path, exist_ok=True)

In [ ]:
from train import run_training

# Start training (parameters can be read from the variables above)
# augment=True if the experiment is not "Pure Dataset"
is_augment = False if experiment == "Pure Dataset" else True

history, model = run_training(
    data_path=data_path, 
    results_folder=results_folder, 
    augment=is_augment,
    epochs=60
)

In [ ]:
from utils.visuals import plot_training_history, plot_confusion_matrix
from sklearn.metrics import classification_report
from data.dataset import LABEL_MAP
import pandas as pd
import numpy as np
import os
import shutil

# - 1. SAVING GRAPHS AND LOGS ---
# results_path is now dynamically defined based on the chosen experiment
print(f" Saving results to: {results_path}")

# Save the trend graph (Accuracy/Loss)
plot_training_history(history, save_path=os.path.join(results_path, 'history_plot.png'))

# Save numeric logs to CSV
hist_df = pd.DataFrame(history.history)
hist_df.to_csv(os.path.join(results_path, 'metrics_log.csv'), index=False)

# - 2. MODEL MANAGEMENT ---
# We use the dynamic checkpoint path defined in train.py (e.g. checkpoints/res_ActorSplit/best_model.keras)
# Compared to your code, the path now reflects the experiment folder
local_model = os.path.join('checkpoints', results_folder, 'best_model.keras')

if os.path.exists(local_model):
    shutil.copy(local_model, os.path.join(results_path, 'best_model_final.keras'))
    print(f" Best model ({local_model}) copied to the results folder.")

# --- 3. FINAL EVALUATION ---
print(" Start final evaluation on the Test Set...")

# We load the test data (X_test and y_test)
X_test = np.load(os.path.join(data_path, 'X_test.npy'))
y_test = np.load(os.path.join(data_path, 'y_test.npy'))

# We generate predictions
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

# Convert y_test labels (strings) to numeric indices using LABEL_MAP
# We handle both the case where y_test is already numeric and the case string
if isinstance(y_test[0], (str, np.str_)):
    y_true = np.array([LABEL_MAP[label] for label in y_test])
else:
    y_true = y_test

# Generate and save the Confusion Matrix
label_names = list(LABEL_MAP.keys())
plot_confusion_matrix(y_true, y_pred, labels=label_names, 
                      save_path=os.path.join(results_path, 'confusion_matrix.png'))

# Generate classification report and save the (Precision, Recall, F1)
report = classification_report(y_true, y_pred, target_names=label_names)
with open(os.path.join(results_path, 'classification_report.txt'), 'w') as f:
    f.write(f"Experiment: {results_folder}\n")
    f.write(f"Dataset: {data_path}\n")
    f.write("-" * 30 + "\n")
    f.write(report)

print(report)
print(" Evaluation completed and results saved!")